In [1]:
import os
import sys
from pathlib import Path

# Add project root to sys.path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import pandas as pd
import numpy as np
from xtheta.data.adapters.hensen import load_hensen_dataset
from xtheta.data.bell_chsh import compute_chsh_variants
from xtheta.data.schema import BellEventSchema

# 06 Hensen (Delft) 2015 Open-Data Audit

This notebook audits the raw Hensen et al. (2015) data and our internal adapter mapping to ensure CHSH S calculation accuracy.

**Scientific Warning:**
Phi_eff is an effective phenomenological parameter only. Without gravitational path, altitude, curvature, or spacetime-baseline metadata, the open Bell/CHSH datasets are not evidence of spacetime-induced X-Theta holonomy. They validate the computational mapping from observed CHSH statistics to effective X-Theta parameters.

## 1. Manual Raw Audit

We inspect the raw file structure directly.

In [2]:
raw_path = Path("../data/open_bell/hensen/raw/bell_open_data.txt")

if not raw_path.exists():
    print(f"[ERROR] Raw data not found at {raw_path}")
    print("Please run: python ../scripts/download_open_data.py --dataset hensen")
else:
    print(f"Raw file path: {raw_path.resolve()}")
    raw_lines = raw_path.read_text(encoding='utf-8').splitlines()
    print(f"\nFirst 10 raw lines:")
    for line in raw_lines[:10]:
        print(line)

Raw file path: C:\workspace\Physics\X-theta\X-theta-framework\xtheta-lab\data\open_bell\hensen\raw\bell_open_data.txt

First 10 raw lines:
2015-06-26 17:24:12.119993,1,2,5445065,1,5671004,1,1,1,10379,10371,11281,13113,0,0,0,0
2015-06-26 17:29:29.620323,1,2,5430292,0,5732887,0,0,1,10375,10369,11437,10714,0,0,0,0
2015-06-26 17:35:04.764434,1,2,5437169,0,5807836,0,1,1,10380,10369,10898,0,0,0,0,0
2015-06-26 17:36:30.324177,1,2,5467363,1,5684689,1,0,1,10380,10367,12780,12680,0,0,0,0
2015-06-26 17:40:26.117068,1,2,5442165,0,5781811,0,0,0,10380,10367,10708,10827,0,0,0,0
2015-06-26 17:42:02.850368,1,2,5423987,1,5688516,0,0,1,10380,10370,10710,0,0,0,0,0
2015-06-26 17:42:09.378446,1,2,5425717,0,5672525,0,0,1,10372,10369,0,0,7616,0,0,0
2015-06-26 17:44:54.196963,1,2,5437094,0,5673314,0,1,0,10376,10368,0,12206,0,0,0,0
2015-06-26 17:47:30.229833,1,2,5424229,0,5691073,0,0,1,10379,10366,0,11430,7607,0,0,0
2015-06-26 18:03:14.427996,1,3,5423690,0,5682413,1,1,1,10379,10368,0,13201,0,0,0,0


## 2. Adapter Audit

We use the `load_hensen_dataset` adapter to parse the data and verify mapping.

In [3]:
if raw_path.exists():
    # Load all data
    df_iter = load_hensen_dataset(str(raw_path))
    df = pd.concat(list(df_iter), ignore_index=True)
    
    print(f"Parsed DataFrame Preview (first 5 rows):")
    display(df.head())
    
    schema = BellEventSchema()
    
    print("\nUnique Alice Settings:", df[schema.alice_setting].unique())
    print("Unique Bob Settings:", df[schema.bob_setting].unique())
    
    print("\nAlice Outcome Counts:")
    print(df[schema.alice_outcome].value_counts())
    
    print("\nBob Outcome Counts:")
    print(df[schema.bob_outcome].value_counts())
    
    print("\nSetting-pair counts:")
    counts = df.groupby([schema.alice_setting, schema.bob_setting]).size()
    print(counts)

Parsed DataFrame Preview (first 5 rows):


,trial_id,timestamp,alice_setting,bob_setting,alice_outcome,bob_outcome,source_file
0,21,2015-06-26 18:45:15.378864,0,0,1,1,bell_open_data.txt
1,31,2015-06-26 19:13:28.649434,0,0,-1,-1,bell_open_data.txt
2,42,2015-06-26 19:55:41.528690,1,1,1,-1,bell_open_data.txt
3,131,2015-06-27 11:05:02.464088,1,1,-1,1,bell_open_data.txt
4,143,2015-06-27 11:38:45.871718,0,0,1,1,bell_open_data.txt



Unique Alice Settings: [0 1]
Unique Bob Settings: [0 1]

Alice Outcome Counts:
alice_outcome
 1    126
-1    119
Name: count, dtype: int64

Bob Outcome Counts:
bob_outcome
-1    127
 1    118
Name: count, dtype: int64

Setting-pair counts:
alice_setting  bob_setting
0              0              53
               1              79
1              0              62
               1              51
dtype: int64


## 3. Correlation and CHSH Audit

Calculate expectations $E(a,b)$ and CHSH variants.

In [4]:
if raw_path.exists():
    df['ab'] = df[schema.alice_outcome] * df[schema.bob_outcome]
    
    # Calculate E(a,b)
    expectations = df.groupby([schema.alice_setting, schema.bob_setting])['ab'].mean()
    
    E00 = expectations.get((0, 0), 0.0)
    E01 = expectations.get((0, 1), 0.0)
    E10 = expectations.get((1, 0), 0.0)
    E11 = expectations.get((1, 1), 0.0)
    
    print(f"E00: {E00:.4f}")
    print(f"E01: {E01:.4f}")
    print(f"E10: {E10:.4f}")
    print(f"E11: {E11:.4f}")

    variants = compute_chsh_variants(E00, E01, E10, E11)
    
    print("\nCHSH Sign Variants:")
    for k, v in variants.items():
        if k not in ['max_abs', 'max_abs_convention']:
            print(f"  {k}: {v:.4f}")
            
    print(f"\nMax Absolute CHSH: {variants['max_abs']:.4f} (Convention: {variants['max_abs_convention']})")

E00: 0.7358
E01: 0.5949
E10: 0.4839
E11: -0.6078

CHSH Sign Variants:
  +++-: 2.4225
  ++-+: 0.2391
  +-++: 0.0169
  -+++: -0.2649

Max Absolute CHSH: 2.4225 (Convention: +++-)
